# NB0b – Notebook-Konventionen
### CAS Information Engineering – Scripting Project

Dieses Notebook definiert verbindliche Konventionen für alle Projektnotebooks.
Ziel: einheitliche Struktur, Nachvollziehbarkeit und Wartbarkeit.

---
> **Geltungsbereich:** NB01–NB10 und alle weiteren Kür-Notebooks.  
> **Ausnahme:** NB01b / NB02b (Simulationsnotebooks) folgen diesen Konventionen nur im Rahmen ihres eingeschränkten Zwecks.  
> **Hinweis:** `99_Datenprovenienz.ipynb` ist manuell auszuführen — nicht Teil von `run_all.sh`.


---
## 1. Zellstruktur

### 1.1 Grundprinzip

Jede inhaltliche Einheit besteht aus **drei Teilen** in dieser Reihenfolge:

```
[MD]   ## Abschnittstitel / Erklärung was folgt
[Code] Implementierung
[Code] Verifikation (head / print / assert)
```

Nie zwei unkommentierte Code-Zellen nacheinander ohne MD-Trenner.

### 1.2 Zellgrösse

- **Max. ~40 Zeilen pro Code-Zelle.** Längere Blöcke aufteilen.
- Eine Code-Zelle = eine klar abgrenzbare Aufgabe.
- Keine mehrseitigen Funktionsdefinitionen ohne MD-Einleitung.

### 1.3 Erste Zelle jedes Notebooks

Immer eine **Titel-Markdown-Zelle** mit:
```markdown
# NB{N} – {Titel}
### CAS Information Engineering – Scripting Project
**Gruppe:** ... | **Datum:** ...

---
Kurze Beschreibung was dieses Notebook tut.
```

### 1.4 Letzte Zellen jedes Notebooks

Immer eine **Abschluss-Code-Zelle** die Ausgabedateien auflistet und validiert,
gefolgt von einer MD-Zelle mit Verweis auf das nächste Notebook.

### 1.5 Kür-Notebooks: Datenladen

Kür-Notebooks laden ihre Datensätze **im ersten Abschnitt des Notebooks, das sie erstmals benötigt** — nicht in NB01.

```
NB01   → lädt nur Pflicht-Datensätze (DS1 Spot-Preise, DS2 Netzlast)
NB06   → lädt BFE GeoPackage + BFS STATPOP + swisstopo in Sektion 1
NB07   → lädt ENTSO-E Grenzflüsse in Sektion 1
NB0x   → jedes weitere Kür-NB lädt seine Datensätze in Sektion 1
```

**Aufbau Sektion 1 in Kür-Notebooks:**
```
[MD]   ## 1. Daten laden
       Hinweis: «Kür-Datensätze werden hier geladen — erstes NB das sie benötigt»
       Tabelle: Datei | Quelle | Pfad | Erstellt in
[Code] Setup (config.json, Pfade, needs_download-Funktion)
[MD]   ### 1.1 <Datensatzname> — Quelle, Methode, Format, Zweck
[Code] Download mit Retry + log_dataindex
[Code] Verifikation (head, shape, range)
[MD]   ### 1.2 <nächster Datensatz> ...
```

Wenn ein Kür-NB einen Datensatz konsumiert, der von einem früheren Kür-NB geladen wurde,
steht in Sektion 1 nur eine Lese-Zelle (kein Download) mit Verweis auf das ladende NB.


---
## 2. Markdown-Konventionen

### Hierarchie

```
# NB-Titel                    ← nur einmal, ganz oben
## 1. Hauptabschnitt          ← Nummeriert, mit führendem ---
### 1.1 Unterabschnitt        ← Nummeriert
**Fetttext**                  ← für Schlüsselbegriffe
`code_inline`                 ← für Variablennamen, Dateinamen
> Blockquote                  ← für wichtige Hinweise / Warnungen
```

### Tabellen

Strukturierte Informationen (Datensätze, Segmente, Ergebnisse) immer als MD-Tabelle,
nicht als Code-Ausgabe.

### Keine Wiederholung von Code-Ausgaben in MD

Nicht: *'Der DataFrame hat 17519 Zeilen'* im MD schreiben — das steht im head().
Stattdessen: Interpretation und Konsequenzen beschreiben.


---
## 3. Code-Konventionen

### 3.1 Kommentare

```python
# ── Abschnittskommentar ──────────────────────────────────────────────────────
# (80 Zeichen, mit ── und trailing ─)

# Inline-Kommentar: erklärt WARUM, nicht WAS
df['price_eur_mwh'] = df['price_eur_mwh'].clip(-500, 3000)  # physikalisch plausibel
```

### 3.2 Variablennamen

| Typ | Konvention | Beispiel |
|-----|------------|----------|
| DataFrame | `df_` Präfix | `df_prices`, `df_load` |
| Dateipfad | `_FILE` Suffix | `PRICES_FILE`, `CLEAN_FILE` |
| Konstante | UPPER_CASE | `FORCE_RELOAD`, `CHARTS_DIR` |
| Hilfsfunktion | snake_case | `needs_download()`, `log_dataindex()` |
| Plot-Objekte | `fig_`, `ax_` | `fig1, ax1 = plt.subplots(...)` |

### 3.3 Importe

Alle Importe in der **ersten Code-Zelle** des Notebooks.
Keine nachträglichen Importe mitten im Notebook ausser bei bedingten/optionalen Paketen.

```python
# ── Importe ──────────────────────────────────────────────────────────────────
import os, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
```

### 3.4 Pfade

Nie hartcodierte absolute Pfade. Immer relativ mit `os.path.join()`:
```python
# Gut
PRICES_FILE = os.path.join(DIR_RAW, 'ch_spot_prices_raw.csv')
# Schlecht
PRICES_FILE = 'C:/Users/patrik/projekt/data/raw/ch_spot_prices_raw.csv'
```


---
## 4. DataFrame-Verifikation (head-Konvention)

### Pflicht: head() nach diesen Operationen

| Auslöser | Verifikations-Code |
|----------|-------------------|
| Laden aus CSV / API | `df.head(3)` + shape + dtypes |
| Nach Bereinigung | `df.head(3)` + Nullwert-Check |
| Nach Feature Engineering | `df[neue_spalten].head(3)` |
| Nach Aggregation/Pivot | `df.head(3)` |
| Nach Merge/Join | `df.head(3)` + shape |

### Pflicht-Template für geladene Datensätze

```python
# ── Verifikation ─────────────────────────────────────────────────────────────
print(f'Shape  : {df_prices.shape}')
print(f'Dtypes : {df_prices.dtypes.to_dict()}')
print(f'Nulls  : {df_prices.isnull().sum().sum()}')
print(f'Range  : {df_prices["price_eur_mwh"].min():.1f} – '
      f'{df_prices["price_eur_mwh"].max():.1f} EUR/MWh')
df_prices.head(3)
```

### Wann kein head() nötig

- Nach reinen Plot-Operationen
- Nach `.to_csv()` Aufrufen
- Nach einfachen Spaltenberechnungen ohne neue Struktur
- Innerhalb von Schleifen / Animationsframes

### Tiefe: 3 Zeilen Standard

`df.head(3)` ist Standard. Bei breiten DataFrames (>10 Spalten): `df.head(3).T` für bessere Lesbarkeit.


---
## 5. Zentrale Konfiguration (config.json)

### 5.1 Zweck — Single Source of Truth

Alle Notebooks lesen ihre Steuerparameter aus `config.json` im Projektverzeichnis.
**Nie** Schalter im Notebook-Code hardcoden — immer über `config.json`.

### 5.2 Struktur & Parameter-Referenz

`config.json` ist in vier Bereiche gegliedert:

```
config.json
├── mode / daten / force_reload / visualisierung   ← global, alle NBs
├── szenarien                                       ← global: NB02 (Pflicht) + NB05/NB06 (Kür)
├── pflicht                                         ← NB01–NB04
└── kuer                                            ← NB06–NB10 (ausschliesslich)
```

#### Globale Parameter

| Key | Default | Beschreibung |
|-----|---------|--------------|
| `mode` | `"data"` | `"data"` = echte Daten (ENTSO-E/BFE) \| `"sim"` = simulierte Daten |
| `eur_chf` | `0.97` | Fixkurs EUR→CHF für das Projekt (März 2026). Alle Berechnungen in EUR; CHF nur informativ. |
| `api_keys.entsoe` | `""` | ENTSO-E API-Key (NB01, NB07) — nie im Notebook hardcoden |
| `daten.start_year` | `2023` | Erstes Jahr Datenabruf (NB01) |
| `daten.end_year` | `"heute"` | Letztes Jahr: Zahl z.B. `2025` oder `"heute"` = aktuelles Jahr |
| `force_reload.spot_prices` | `false` | DS1 ENTSO-E Preise neu laden (NB01) |
| `force_reload.netzlast` | `false` | DS2 ENTSO-E Netzlast neu laden (NB01) |
| `force_reload.crossborder` | `false` | DS3 Grenzflüsse neu laden (NB01, optional) |
| `force_reload.bfe` | `false` | DS4 BFE GeoPackage neu laden (NB01) |
| `force_reload.clean` | `false` | Bereinigte Preise neu erzeugen (NB02 Sek. 1) |
| `force_reload.wirtschaft` | `false` | Wirtschaftlichkeits-CSV neu erzeugen (NB02 Sek. 4) |
| `force_reload.szenarien` | `false` | Szenarien-CSV neu erzeugen (NB02 Sek. 5) |
| `force_reload.import_export` | `false` | Import/Export-Analyse neu erzeugen (NB02 Sek. 6) |
| `force_reload.spread_ts` | `false` | Spread-Zeitreihe neu erzeugen (NB02 Sek. 7) |
| `force_reload.bfs` | `false` | BFS STATPOP Kantone neu laden (NB06) |
| `force_reload.zonenbilanzen` | `false` | Zonenbilanzen-CSV neu erzeugen (NB06) |
| `visualisierung.symlog_linthresh` | `10000` | Symlog linearer Bereich Amortisationsplots |
| `visualisierung.output_dpi` | `150` | DPI für gespeicherte Charts |

#### Szenarien (global — NB02 Pflicht + NB05/NB06 Kür)

| Key | Default | Beschreibung |
|-----|---------|--------------|
| `szenarien.gleichzeitigkeit_aktiv` | `"realistisch"` | Aktives Szenario: `"pessimistisch"` \| `"realistisch"` \| `"optimistisch"` |
| `szenarien.ch_spitzenlast_gw` | `10.5` | CH Systemspitzenlast Referenzwert [GW] |
| `szenarien.optionen.<x>.rate` | 0.15/0.40/0.70 | Gleichzeitigkeitsrate je Szenario |
| `szenarien.optionen.<x>.n_privat_YYYY` | — | Anzahl Privat-Einheiten im Szenario-Jahr |
| `szenarien.optionen.<x>.n_gewerbe_YYYY` | — | Anzahl Gewerbe-Einheiten im Szenario-Jahr |
| `szenarien.optionen.<x>.n_industrie_YYYY` | — | Anzahl Industrie-Einheiten im Szenario-Jahr |

> Nur `gleichzeitigkeit_aktiv` umschalten — die `optionen`-Werte beschreiben reale Marktannahmen und sollten nicht verändert werden.

#### Pflicht-Parameter (NB01–NB04)

| Key | Default | Beschreibung |
|-----|---------|--------------|
| `pflicht.simulation.charge_quantile` | `0.25` | Tages-Quantil Ladeentscheidung (p25) |
| `pflicht.simulation.discharge_quantile` | `0.75` | Tages-Quantil Einspeiseentscheidung (p75) |
| `pflicht.simulation.soc_min_pct` | `0.05` | Minimaler SoC — Tiefentladeschutz |
| `pflicht.simulation.soc_max_pct` | `0.95` | Maximaler SoC — Überladesch. |
| `pflicht.simulation.efficiency_roundtrip` | `0.92` | Round-Trip-Wirkungsgrad Li-Ion |
| `pflicht.wirtschaftlichkeit.capex_eur_kwh` | 400/300/220/180 | CAPEX in EUR/kWh je Segment |
| `pflicht.wirtschaftlichkeit.opex_rate` | `0.015` | Jährliche Betriebskosten (1.5% des CAPEX) |
| `pflicht.wirtschaftlichkeit.lifetime_j` | `12` | Li-Ion Laufzeit bis ~80% Kapazitätserhalt. **Ziel-ROI wird nicht gespeichert** — jedes NB berechnet lokal: `ZIEL_ROI = round(100/LIFETIME, 2)` |
| `pflicht.langzeit.lifetime_long_j` | `20` | Langzeit-Chart Horizont (Chart 1e) [Jahre] |

#### Kür-Parameter (NB06–NB10, ausschliesslich)

| Key | Default | Beschreibung |
|-----|---------|--------------|
| `kuer.raeumlich.bvi_gewichte.netzimbalance` | `0.5` | BVI-Gewicht Netzimbalance (NB06) |
| `kuer.raeumlich.bvi_gewichte.engpassnaehe` | `0.3` | BVI-Gewicht Engpassnähe (NB06) |
| `kuer.raeumlich.bvi_gewichte.saisonal` | `0.2` | BVI-Gewicht Saisonalität (NB06) |
| `kuer.crossborder.nachbarn` | `["DE","AT","IT","FR"]` | Grenzländer für NB07 |
| `kuer.markt.spread_breakeven_privat_eur_mwh` | `30` | Spread-Trigger Privat-Investition (NB08) |
| `kuer.markt.capex_lernrate_pct_pro_jahr` | `10` | CAPEX-Lernrate [%/Jahr] (NB08) |
| `kuer.markt.capex_ziel_privat_eur_kwh` | `250` | CAPEX-Trigger Privat-Investition (NB08) |
| `kuer.dispatch.da_optimal_enabled` | `true` | DA-optimalen Dispatcher aktivieren (NB10) |

### 5.3 Config-Loader (erste Code-Zelle jedes Notebooks)

```python
import os, json as _json
with open('config.json') as _f:
    CFG = _json.load(_f)

MODE         = CFG['mode']
FORCE_RELOAD = CFG['force_reload']

# Pflicht-NBs (NB01–NB04):
_sim  = CFG['pflicht']['simulation']
_w    = CFG['pflicht']['wirtschaftlichkeit']
LIFETIME     = _w['lifetime_j']
ZIEL_ROI     = round(100 / LIFETIME, 2)  # 1/lifetime_j — nie aus config lesen

# Szenarien (NB02 + Kür):
SZ_AKTIV = CFG['szenarien']['gleichzeitigkeit_aktiv']
RATE     = CFG['szenarien']['optionen'][SZ_AKTIV]['rate']
SZ_OPT   = CFG['szenarien']['optionen'][SZ_AKTIV]

# Kür-NBs (NB06–NB10):
# kuer_param = CFG['kuer']['raeumlich']  # etc.
```

### 5.3 API-Keys

Alle API-Keys gehören in `config.json → api_keys`, **nie** im Notebook-Code hardcoden:

```json
"api_keys": {
  "entsoe": "<dein-key>"
}
```

Nutzung im Notebook: `CFG.get('api_keys', {}).get('entsoe', '')`


---
## 5b. Datenzeitraum: Empfehlungen & Stolperfallen

### Empfehlungen

- **`end_year: 'heute'`** verwenden — so läuft das Skript jedes Jahr neu ohne Änderung
- **Mindestens 2 vollständige Kalenderjahre** laden für saisonale Analyse (NB2, Chart 5)
- **Jährlicher Re-Run** empfohlen: Chart 7 zeigt ob Spread gestiegen/gefallen ist
- Nach Änderung von `start_year`/`end_year`: alle `force_reload`-Keys auf `true` setzen
  und NB1–NB3 komplett neu ausführen (`run_all.sh`)

### Stolperfallen

| Problem | Ursache | Lösung |
|---|---|---|
| ENTSO-E gibt 503 zurück | Server temporär überlastet | 15–30 Min warten, dann erneut |
| Lückenhafter Datensatz | Einzelne Jahre nicht verfügbar | In NB1 Abschlusskontrolle prüfen |
| `n_years = 1` statt 2 | `end_year` liegt in der Zukunft | ENTSO-E liefert nur bis heute minus ~2 Tage |
| Simulation gibt `NaN` | Zu wenig Daten für Quantil-Berechnung | Mindestens 3 Monate pro Jahr nötig |
| Spread-Chart springt | Fehlende Monate im Datensatz | `force_reload.clean: true` und NB2 neu |
| Alte CSVs nach Zeitraumänderung | `force_reload` noch auf `false` | Alle relevanten Keys auf `true` |

### Besonderheiten ENTSO-E API

- Day-Ahead-Preise werden typisch **einen Tag nach Handelsschluss** verfügbar
- Für `end_year: 'heute'` wird nur bis zum Vortag geladen — ENTSO-E hat keine Intraday-Daten
- Lücken über **Feiertage** (besonders Ostern, Weihnachten) sind normal und werden interpoliert
- Die ENTSO-E API hat **Rate Limits**: zu viele Requests in kurzer Zeit → 429-Fehler
  → jahresweiser Download mit `_fetch_prices_year()` umgeht das Problem
- Wechsel der **Zeitzone** (CET↔CEST) kann zu doppelten oder fehlenden Stunden führen
  → UTC-Normierung in NB2 bereinigt das


---
## 5c. Ergebnistransfer zwischen Notebooks (transfer.json)

### Zweck — berechnete Outputs, keine config-Redundanz

`transfer.json` enthält ausschliesslich **Werte, die ein Notebook berechnet und ein folgendes benötigt**. Was in `config.json` steht (Parameter, Schalter), wird nie nach `transfer.json` kopiert.

> **Faustregel:** `config.json` schreibt der User. `transfer.json` schreiben die Notebooks.

### Schema — vollständige Referenz

| Sektion | Geschrieben von | Gelesen von | Inhalt |
|---------|----------------|-------------|--------|
| `datenzeitraum` | **NB01** | NB02–NB15 | `start_date`, `end_date`, `n_years` — aus tatsächlich geladenen Preisdaten |
| `simulation` | **NB02** | NB04, NB05, NB09, NB14, NB15 | Spread-Kennzahlen; ROI, `net_annual`, `capex`, `payback_years` je Segment |
| `dispatch_optimierung` | **NB10** | NB05 | DA-optimal vs. reaktiv: `roi_reaktiv_pct`, `roi_da_optimal_pct`, `delta_pct` je Segment |
| `eigenverbrauch` | **NB13** | NB14 | `roi_ev_privat_pct`, `payback_ev_privat_j`, `jahresersparnis_eur` |
| `hybrid_simulation` | **NB15** | NB14 | `roi_arb/ev/hyb/hybo`, `be_*` je Segment |
| `produkt` | **NB14** | NB05 | `roi_arb_pct`, `roi_ev_pct`, `roi_kombi_pct`, `payback_kombi_j` |

### Ausführungsreihenfolge für vollständiges transfer.json

```
NB01 → NB02 → NB10 → NB13 → NB15 → NB14
```

Jedes Notebook kann einzeln neu ausgeführt werden — es überschreibt nur seine eigene Sektion.

### Laden (alle Notebooks — robust gegen leere/fehlende Datei)

```python
import os as _os, json as _json
_tf_path = 'transfer.json'
_tf = _json.loads(open(_tf_path).read() or '{}') \
      if _os.path.exists(_tf_path) and _os.path.getsize(_tf_path) > 0 else {}

# Werte lesen — immer mit Fallback:
TF_N_YEARS = _tf.get('datenzeitraum', {}).get('n_years', None)
TF_ECON    = _tf.get('simulation', {}).get('wirtschaftlichkeit', {})
```

### Schreiben (nur Notebooks die Transferwerte erzeugen)

```python
_tf = _json.loads(open(_tf_path).read() or '{}') \
      if _os.path.exists(_tf_path) and _os.path.getsize(_tf_path) > 0 else {}
_tf['simulation'] = { ... }          # nur eigene Sektion — Rest bleibt erhalten
with open(_tf_path, 'w') as _f:
    _json.dump(_tf, _f, indent=2, ensure_ascii=False)
```

### Regeln

- Nie manuell editieren — Werte stammen immer aus Notebook-Runs
- Keine `config.json`-Werte spiegeln (kein `kuer_aktiv`, keine Parameter)
- `getsize > 0`-Guard immer verwenden — verhindert Crash bei leerer Datei
- Fehlende Sektionen immer mit `.get('sektion', {})` und Fallback lesen


---
## 5d. Chart-Naming-Konvention

### Grundprinzip

Alle Charts in **einem Verzeichnis**: `output/charts/<SZ_AKTIV>/`

```python
CHARTS_DIR = os.path.join('output', 'charts', SZ_AKTIV)
```

Kein separates `output/kuer/` oder `animation/`. Das Präfix im Dateinamen ersetzt die Ordnerhierarchie.

### Namensschema

```
{präfix}_{beschreibung}.{ext}
```

| Präfix | NB | Beispiel |
|--------|-----|---------|
| `nb03_` | NB03 Pflicht | `nb03_roi.png` |
| `kuer_nb06_` | NB06 Räumliche Analyse | `kuer_nb06_karte_engpaesse.png` |
| `kuer_nb07_` | NB07 Cross-Border | `kuer_nb07_import_export.png` |
| `kuer_nb08_` | NB08 Marktdynamik | `kuer_nb08_spread_zeitreihe.png` |
| `kuer_nb08a_` | NB08a Animationen (.gif) | `kuer_nb08a_anim_B_4panel.gif` |
| `kuer_nb09_` | NB09 Revenue Stacking | `kuer_nb09_revenue_stacking.png` |
| `kuer_nb10_` | NB10 Dispatch | `kuer_nb10_dispatch_vergleich.png` |
| `kuer_nb11_` | NB11 Technologievergleich | `kuer_nb11_radar.png` |
| `kuer_nb12_` | NB12 Alternative Speicher | `kuer_nb12_positionierung.png` |
| `kuer_nb13_` | NB13 Eigenverbrauch | `kuer_nb13_amortisation.png` |
| `kuer_nb14_` | NB14 Produktsteckbrief | `kuer_nb14_kennzahlen.png` |
| `kuer_nb15_` | NB15 Komb. Simulation | `kuer_nb15_roi_vergleich.png` |
| `nb99_` | NB99 Datenprovenienz | `nb99_prov_pipeline.png` |

### Vollständige Dateiliste NB03 (Pflicht)

`nb03_wirtschaftlichkeit.png`, `nb03_amortisation.png`, `nb03_roi.png`, `nb03_erloese_kwh.png`, `nb03_capex_ertrag.png`, `nb03_langzeit.png`, `nb03_heatmap_preis.png`, `nb03_heatmap_volatilitaet.png`, `nb03_tagesprofil.png`, `nb03_tagesprofil_einzel.png`, `nb03_netzentlastung.png`, `nb03_spitzenlast.png`, `nb03_spitzenlast_reduktion.png`, `nb03_saisonal_tagesprofil.png`, `nb03_saisonal_{saison}.png` (je Saison), `nb03_saisonal_roi.png`, `nb03_spread_monatlich.png`, `nb03_negativpreise.png`

### NB08a Animationen

`kuer_nb08a_anim_A_{h:02d}h.gif` (je Tageszeit), `kuer_nb08a_anim_B_4panel.gif`, `kuer_nb08a_anim_C_spread.gif`


---
## 5e. Visualisierungsfarben (aus config.json)

Alle Projektfarben sind SSOT in `config.json → visualisierung.farben`. Notebooks lesen sie daraus — nie hardcoden.

```python
# Laden in jeder Setup-Zelle:
_viz = CFG.get('visualisierung', {})
BG_DARK    = _viz.get('bg_dark',   '#0d1117')
BG_PANEL   = _viz.get('bg_panel',  '#141414')
C_PRICE    = _viz.get('c_price',   '#FFA726')
C_LOAD     = _viz.get('c_load',    '#66BB6A')
C_CHARGE   = _viz.get('c_charge',  '#1565C0')
C_FEED     = _viz.get('c_feed',    '#B71C1C')
SEG_COLORS = _viz.get('seg_colors', ['#42A5F5','#66BB6A','#FFA726','#EF5350'])
```

| Variable | Hex | Verwendung |
|----------|-----|------------|
| `BG_DARK` | `#0d1117` | Figure-Hintergrund |
| `BG_PANEL` | `#141414` | Axes-Hintergrund |
| `C_PRICE` | `#FFA726` | Preisverlauf |
| `C_LOAD` | `#66BB6A` | Netzlast |
| `C_CHARGE` | `#1565C0` | Ladevorgang |
| `C_FEED` | `#B71C1C` | Einspeisung |
| `SEG_COLORS[0]` | `#42A5F5` | Privat 10 kWh |
| `SEG_COLORS[1]` | `#66BB6A` | Gewerbe 100 kWh |
| `SEG_COLORS[2]` | `#FFA726` | Industrie 1 MWh |
| `SEG_COLORS[3]` | `#EF5350` | Utility 10 MWh |

> Wer eine Farbe ändern möchte: einmal in `config.json` anpassen — alle Charts aktualisieren sich beim nächsten Run automatisch.


---
## 6. Datenprotokoll (dataindex.csv)

**Jede** erzeugte oder geladene Datei wird in `dataindex.csv` geloggt:

```python
log_dataindex(
    filename   = 'ch_spot_prices_raw.csv',
    source_url = 'https://transparency.entsoe.eu',
    local_path = PRICES_FILE,
    data_type  = 'raw',           # raw | processed | intermediate
    rows       = len(df_prices),
    size_kb    = os.path.getsize(PRICES_FILE) / 1024,
    note       = 'ENTSO-E API, entsoe-py, 2023-2024'
)
```

Nie manuell in `dataindex.csv` schreiben. Immer über `log_dataindex()`.


---
## 7. Visualisierungskonventionen

### Farbschema (durchgängig)

```python
# Farben aus config.json laden (SSOT — nie hardcoden):
_viz = CFG.get('visualisierung', {})
BG_DARK    = _viz.get('bg_dark',    '#0d1117')
BG_PANEL   = _viz.get('bg_panel',   '#141414')
C_PRICE    = _viz.get('c_price',    '#FFA726')
C_LOAD     = _viz.get('c_load',     '#66BB6A')
C_CHARGE   = _viz.get('c_charge',   '#1565C0')
C_FEED     = _viz.get('c_feed',     '#B71C1C')
SEG_COLORS = _viz.get('seg_colors', ['#42A5F5','#66BB6A','#FFA726','#EF5350'])
```
> Farben zentral in `config.json → visualisierung.farben` — siehe Sektion 5e.

### Pflicht pro Chart

1. `fig.patch.set_facecolor(BG_DARK)` + `ax.set_facecolor(BG_PANEL)`
2. `ax.tick_params(colors='#bbbbbb')`
3. `ax.set_title(..., color='white', fontweight='bold')`
4. `plt.savefig(path, dpi=150, bbox_inches='tight', facecolor=BG_DARK)`
5. `plt.show()` gefolgt von `plt.close()` bei Einzelplots

### Einzelplots

Jeder Komposit-Chart (mehrere Panels) hat zusätzlich Einzelplot-Exports in `charts/` mit Suffix `_single.png`. Grösse Einzelplot: mindestens `(10, 6)`.

### 7.2 Panel-Einzelplots

Jedes mehrteilige Chart speichert **zusätzlich jeden Panel als eigenständiges Bild** — direkt im Anschluss an den Gesamt-Chart in derselben Code-Zelle oder der unmittelbar folgenden.

**Gilt für alle Notebooks** (Pflicht und Kür). Saisonale Charts speichern alle Saisons, nicht nur die Extremwerte.

**Namenskonvention:** `chart<N><panel>_<beschreibung>.png` — z.B. `chart5a_frühling_single.png`, `chart7b_negativpreise.png`.

```python
# Muster nach dem Gesamt-Chart:
for fname, ax_src, title in [
    ('chartXa_panel1.png', axes[0], 'Panel 1'),
    ('chartXb_panel2.png', axes[1], 'Panel 2'),
]:
    fig_e, ax_e = plt.subplots(figsize=(10, 5))
    fig_e.patch.set_facecolor('#0d1117')
    # Inhalt aus ax_src reproduzieren
    plt.savefig(os.path.join(CHARTS_DIR, fname), dpi=150, ...)
    plt.close()
```


---
## 8. Abschlussblock (jedes Notebook)

Die letzte Code-Zelle jedes Notebooks:

```python
# ── Abschlusskontrolle ────────────────────────────────────────────────────────
print('=' * 60)
print(f'NB{N} abgeschlossen')
print('=' * 60)
EXPECTED_FILES = [
    (PRICES_FILE, 10,   'ENTSO-E Preise'),
    (CLEAN_FILE,  10,   'Bereinigte Preise'),
]
all_ok = True
for path, min_kb, desc in EXPECTED_FILES:
    ok = os.path.exists(path) and os.path.getsize(path)/1024 >= min_kb
    icon = '✅' if ok else '❌'
    print(f'  {icon} {desc:<35} {path}')
    if not ok: all_ok = False
print()
print('→ Weiter mit NB{N+1}' if all_ok else '→ Fehler beheben vor NB{N+1}'  # Pflicht: NB01-NB04 | Kür: NB05-NB10)
```


---
## 9. Review-Checkliste

Vor der Abgabe jedes Notebook gegen diese Liste prüfen:

- [ ] Alle Zellen von oben nach unten ohne Fehler ausführbar (`Kernel → Restart & Run All`)
- [ ] Erste Zelle: Titel-MD mit Gruppenname und Datum
- [ ] Letzte Zellen: Abschlusskontrolle + Verweis auf nächstes NB
- [ ] Jeder neu geladene/erzeugte DataFrame hat eine head()-Verifikationszelle
- [ ] `FORCE_RELOAD`-Dict vorhanden und alle Keys auf `False`
- [ ] Alle erzeugten Dateien in `dataindex.csv` geloggt
- [ ] Keine absoluten Pfade
- [ ] Alle Charts gespeichert und `plt.close()` aufgerufen
- [ ] Keine unnötigen `print()`-Ausgaben aus Debugging-Sessions
- [ ] Zellen-Outputs vorhanden (Notebook nicht leer/nicht-ausgeführt einreichen)
- [ ] `config.json` korrekt gelesen (nie MODE/FORCE_RELOAD hardcoden)
- [ ] Farben aus `CFG['visualisierung']` geladen — nicht hardcoded
- [ ] Chart-Namen folgen Konvention: `nb03_*.png` / `kuer_nb{NN}_*.png` (Sektion 5d)
- [ ] `transfer.json` mit `getsize > 0`-Guard geladen; nur eigene Sektion geschrieben (Sektion 5c)
- [ ] Kür-NBs: Szenario-Outputs in `output/kuer/<szenario>/` gespeichert
- [ ] `intermediate/<szenario>/` für szenario-abhängige Dateien verwendet


---
## 10. Musterzelle: DataFrame laden

So sieht eine regelkonforme Lade- + Verifikationssequenz aus:


In [1]:
# ── Beispiel: Regelkonformes Laden + Verifikation ────────────────────────────
# (Diese Zelle ist nur zur Demonstration — nicht ausführen)

# Laden
# df_prices = pd.read_csv(PRICES_FILE, parse_dates=['timestamp'])
# df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], utc=True)

# Verifikation — separate Zelle
# print(f'Shape   : {df_prices.shape}')
# print(f'Zeitraum: {df_prices["timestamp"].min()} – {df_prices["timestamp"].max()}')
# print(f'Nulls   : {df_prices.isnull().sum().sum()}')
# print(f'Range   : {df_prices["price_eur_mwh"].min():.1f} – '
#       f'{df_prices["price_eur_mwh"].max():.1f} EUR/MWh')
# df_prices.head(3)

print('Musterzelle — nur zur Demonstration.')


Musterzelle — nur zur Demonstration.
